# HostWise - Track B: Exploratory Data Analysis (five cities)

**Cities:** Barcelona, London, Madrid, Paris, Rome (New York excluded).

**Goal:** find which listing characteristics are related to the nightly price **within each city**, and how strongly, to decide the Day 2 feature set. This notebook explores relationships only; it does not clean data or train models.

**Notebook guide**

| Section | Question |
|---|---|
| B0 | Which listings does the EDA use, and how are prices compared across currencies? |
| B1 | How are nightly prices distributed within each city? |
| B2 | How different are room types in price? |
| B3 | How does price change with guest capacity? |
| B4 | Do bedrooms, beds and bathrooms add information beyond capacity? |
| B5 | Does the property type matter, and can it be grouped? |
| B6 | How much does location explain? |
| B7 | Do host characteristics relate to price? |
| B8 | Which amenities are associated with higher prices? |
| B9 | Do reviews relate to price? |
| B10 | Which features explain the most price variance? |

**Rules used throughout:** prices are never compared across cities in raw currency; every price relationship is computed within a city or with a currency-free relative measure (B0). Associations are not causal effects.

## 0. Setup

The setup cells are identical in every HostWise notebook, so all tracks use the same data and the same population definitions.

| Name | Definition |
|---|---|
| `df_all` | All listings in `hostwise_5cities.csv` (five cities) |
| `df_active` | `source == "city scrape"`: listings present in the current scrape |
| `df_priced` | `df_active` with a parseable price |
| `df_scope` | `df_priced` with `minimum_nights <= 29` (short-term scope, pending team approval) |

Conventions: the parsed price is `price_num` and the log target is `log_price` (`log1p`, inverse `expm1`). Prices are in local currency (London GBP, other cities EUR).

Track B adds one population, built in section B0:

| Name | Definition |
|---|---|
| `df_eda` | `df_scope` inside the per-city log-IQR price fence (k = 3) with a floor of 10 - for exploration only |

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [ ]:
DATA_CANDIDATES = [
    Path("../data/processed/hostwise_5cities.csv"),
    Path("data/processed/hostwise_5cities.csv"),
    Path("hostwise_5cities.csv"),
    Path("Airbnb/hostwise_5cities.csv"),
]
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

CITY_ORDER = ["london", "paris", "rome", "madrid", "barcelona"]
CURRENCY = {"london": "GBP", "paris": "EUR", "rome": "EUR",
            "madrid": "EUR", "barcelona": "EUR"}

MAX_SHORT_STAY_NIGHTS = 29   # short-term scope threshold (pending team approval)
PRICE_FLOOR = 10             # data-validity floor in local currency (pending team approval)
FENCE_K = 3                  # log-IQR fence multiplier for price outliers

In [ ]:
data_file = next((path for path in DATA_CANDIDATES if path.exists()), None)
if data_file is None:
    raise FileNotFoundError("hostwise_5cities.csv not found - update DATA_CANDIDATES.")

df_all = pd.read_csv(data_file, low_memory=False)

# Validation: the file must contain exactly the five project cities (no New York).
assert set(df_all["city"].unique()) == set(CITY_ORDER), sorted(df_all["city"].unique())
print(f"df_all: {df_all.shape[0]:,} listings x {df_all.shape[1]} columns")
print(df_all["city"].value_counts().reindex(CITY_ORDER).to_string())

In [ ]:
# Derived target columns. The original `price` column is never modified.
df_all["price_num"] = pd.to_numeric(
    df_all["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce"
)
df_all["log_price"] = np.log1p(df_all["price_num"])

# Analysis populations (identical definitions in every HostWise notebook)
df_active = df_all[df_all["source"] == "city scrape"]
df_priced = df_active[df_active["price_num"].notna()]
df_scope = df_priced[df_priced["minimum_nights"] <= MAX_SHORT_STAY_NIGHTS]

print(f"df_active: {len(df_active):,} | df_priced: {len(df_priced):,} | df_scope: {len(df_scope):,}")

## B0. EDA population and relative price measures

**Research question:** Which listings should the EDA describe, and how can price relationships be compared across cities with different currencies and price levels?

**Purpose:** The earlier Track B used all active priced listings (168,725), including 30+ night rentals that are outside the proposed short-term scope, and it hid prices above 1,000 from the charts. Here the EDA uses the modelling population (`df_scope`) and removes only the extreme prices flagged by the outlier rule proposed in Track C (C9). The fences are computed for exploration only; Day 2 recomputes them on the training data.

To compare cities, prices are expressed **relative to a city benchmark** on the log scale. A difference of 0.10 in log price is about +10.5% in price, in any currency:

| Measure | Definition | Removes |
|---|---|---|
| `rel_log` | `log_price` minus the city median | City price level and currency |
| `rel_log_entire` | Entire homes: `log_price` minus the city median of entire homes | City level and room type |
| `rel_log_size` | Entire homes: `log_price` minus the median for the same city and guest capacity | City level, room type and size |

`rel_log_size` is the main measure for features such as amenities or reviews, because it compares listings of the same type and size in the same city.

In [ ]:
def log_iqr_fence(prices: pd.Series, k: float = FENCE_K) -> pd.Series:
    log_prices = np.log(prices)
    q1, q3 = log_prices.quantile([0.25, 0.75])
    iqr = q3 - q1
    return pd.Series({"lower": np.exp(q1 - k * iqr), "upper": np.exp(q3 + k * iqr)})


price_fences = df_scope.groupby("city")["price_num"].apply(log_iqr_fence).unstack().reindex(CITY_ORDER)
price_fences

In [ ]:
lower_fence = np.maximum(df_scope["city"].map(price_fences["lower"]), PRICE_FLOOR)
upper_fence = df_scope["city"].map(price_fences["upper"])

df_eda = df_scope[df_scope["price_num"].between(lower_fence, upper_fence)].copy()
print(f"df_eda: {len(df_eda):,} of {len(df_scope):,} scope listings ({len(df_eda) / len(df_scope):.2%})")

In [ ]:
df_eda["city"].value_counts().reindex(CITY_ORDER).rename("listings").to_frame()

In [ ]:
df_eda["rel_log"] = df_eda["log_price"] - df_eda.groupby("city")["log_price"].transform("median")
df_eda["guests"] = df_eda["accommodates"].clip(upper=10)
df_eda["is_entire"] = df_eda["room_type"] == "Entire home/apt"

entire = df_eda[df_eda["is_entire"]]
df_eda["rel_log_entire"] = entire["log_price"] - entire.groupby("city")["log_price"].transform("median")
df_eda["rel_log_size"] = entire["log_price"] - entire.groupby(["city", "guests"])["log_price"].transform("median")

df_eda[["rel_log", "rel_log_entire", "rel_log_size"]].describe()

In [ ]:
def to_pct(log_difference):
    """Convert a difference in log price into a percentage price difference."""
    return (np.exp(log_difference) - 1) * 100


def median_premium_pct(frame, group, value="rel_log"):
    """Median relative price (in %) per city and group."""
    table = frame.groupby(["city", group], observed=True)[value].median().unstack()
    return to_pct(table).reindex(CITY_ORDER)


def spearman_by_city(frame, feature, target="log_price"):
    """Spearman rank correlation between a feature and the target, computed within each city."""
    result = {}
    for city, group in frame.groupby("city"):
        pair = group[[feature, target]].dropna()
        result[city] = pair[feature].rank().corr(pair[target].rank())
    return pd.Series(result, name=feature).reindex(CITY_ORDER)


def variance_explained(frame, feature, target="log_price"):
    """Within-city eta squared: share of log-price variance explained by the group means of a feature."""
    data = frame[frame[feature].notna()]
    city_mean = data.groupby("city")[target].transform("mean")
    group_mean = data.groupby(["city", feature], observed=True)[target].transform("mean")
    ss_between = ((group_mean - city_mean) ** 2).groupby(data["city"]).sum()
    ss_total = ((data[target] - city_mean) ** 2).groupby(data["city"]).sum()
    return (ss_between / ss_total).rename(feature).reindex(CITY_ORDER)

### Findings

- `df_scope` has 153,812 listings. The fences and the floor remove only a few hundred extreme prices: in Track C, 432 scope listings were flagged by the 3x log-IQR fence, and London's lower fence (about 4 GBP) required the floor of 10.
- The EDA now describes the same short-term population the model will be trained on, instead of all 168,725 active priced listings (which include 30+ night rentals).

> **Pending execution:** record the `df_eda` size and its per-city counts (expected about 153,380).

### Conclusion

- All EDA statistics below are computed on `df_eda`. They describe the short-term market and exclude only the most extreme prices.
- The relative measures make cross-city comparisons valid without currency conversion.

## B1. Price distribution within each city

**Research question:** How are nightly prices distributed in each city, and is a log transformation of the target justified?

**Purpose:** The shape of the target decides the modelling target and the error metric. The earlier charts cut the x-axis at 1,000, which hid 3,751 of 168,725 listings and hid the tail shape. A log-scale axis shows the full range instead.

In [ ]:
price_summary = df_eda.groupby("city")["price_num"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).reindex(CITY_ORDER)
price_summary["currency"] = price_summary.index.map(CURRENCY)
price_summary

In [ ]:
pd.DataFrame({
    "skew_price": df_eda.groupby("city")["price_num"].skew(),
    "skew_log_price": df_eda.groupby("city")["log_price"].skew(),
}).reindex(CITY_ORDER)

In [ ]:
log_quartiles = df_eda.groupby("city")["log_price"].quantile([0.1, 0.25, 0.75, 0.9]).unstack()

price_dispersion = pd.DataFrame({
    "p75 / p25 price ratio": np.exp(log_quartiles[0.75] - log_quartiles[0.25]),
    "p90 / p10 price ratio": np.exp(log_quartiles[0.9] - log_quartiles[0.1]),
}).reindex(CITY_ORDER)
price_dispersion

In [ ]:
g = sns.displot(
    data=df_eda, x="price_num", col="city", col_order=CITY_ORDER, col_wrap=3,
    bins=50, log_scale=True, height=3, facet_kws={"sharex": False, "sharey": False},
)
g.set_axis_labels("Nightly price, log scale (local currency)", "Listings")
g.set_titles("{col_name}")
g.figure.suptitle("Nightly price distribution per city (df_eda)", y=1.03)
g.savefig(FIG_DIR / "b1_price_distribution.png", dpi=150)
plt.show()

### Findings

- **Median nightly price in the short-term scope (`df_scope`, Track C):** Barcelona 255 EUR, Paris 230 EUR, London 183 GBP, Rome 166 EUR, Madrid 131 EUR.
- **Barcelona's median rises from 181 EUR (all active priced listings) to 255 EUR (short-term scope)** because the scope removes its many cheaper 30+ night listings (Track C, C7). The previous EDA population understated Barcelona's short-term price level.
- In all active priced listings, the mean was 35-58% above the median in every city: the raw price is strongly right-skewed.

> **Pending execution:** record the per-city skew of `price_num` vs `log_price` and the price-ratio table. Expected: skew falls close to 0 on the log scale. The p90/p10 ratio shows which city has the widest price range, i.e. where a fixed error in % is hardest to achieve.

### Conclusion

- **Evidence-based:** model `log_price` (`log1p`) and convert predictions back with `expm1`. Errors on the log scale are relative errors, which makes cities with different price levels comparable.
- **Proposed metric (pending team approval):** MAE on the log scale per city, macro-averaged across cities, plus MAE in local currency per city for the presentation.

## B2. Room type

**Research question:** How much does the nightly price differ between room types within each city?

**Purpose:** If room types differ strongly in price, `room_type` is a core feature and other relationships must be checked within room type, otherwise a feature that simply proxies for "private room" will look important.

In [ ]:
pd.crosstab(df_eda["city"], df_eda["room_type"]).reindex(CITY_ORDER)

In [ ]:
df_eda.groupby(["city", "room_type"])["price_num"].median().unstack().reindex(CITY_ORDER)

In [ ]:
median_log_by_room = df_eda.groupby(["city", "room_type"])["log_price"].median().unstack()
room_vs_entire_pct = to_pct(median_log_by_room.sub(median_log_by_room["Entire home/apt"], axis=0)).reindex(CITY_ORDER)
room_vs_entire_pct

In [ ]:
ROOM_ORDER = ["Entire home/apt", "Private room", "Hotel room", "Shared room"]

g = sns.catplot(
    data=df_eda, x="room_type", y="price_num", col="city", col_order=CITY_ORDER, col_wrap=3,
    order=ROOM_ORDER, kind="box", showfliers=False, sharey=False, height=3.2,
)
g.set(yscale="log")
g.set_xticklabels(rotation=40)
g.set_axis_labels("", "Nightly price, log scale (local currency)")
g.set_titles("{col_name}")
g.tight_layout()
g.savefig(FIG_DIR / "b2_price_by_room_type.png", dpi=150)
plt.show()

### Findings

- Among active listings, entire homes are 77.0% of supply (134,741), private rooms 22.2% (38,775), hotel rooms 785 and shared rooms 621 (Track A, A3).
- Track C (C8) already showed that private rooms are much cheaper: they are the main reason listings with missing `bedrooms` have a lower median price.

> **Pending execution:** record the room-type counts per city and the `room_vs_entire_pct` table (the % price difference of each room type vs entire homes in the same city). Check whether hotel and shared rooms have enough listings per city (fewer than ~100 makes their medians unreliable).

### Conclusion

- `room_type` is a required feature; most later analyses use entire homes only (`rel_log_size`) so that room type does not confound the result.
- **Pending team decision:** keep or drop Hotel and Shared rooms. Each is under 1% of active listings. Keeping them costs little, but their errors will be large and should be reported separately.

## B3. Guest capacity

**Research question:** How does the nightly price change with the number of guests a listing accommodates, within each city?

**Purpose:** Capacity is the main size feature available to a new host. The shape of the relationship (linear, flattening, noisy at the top) decides whether `accommodates` can be used as-is or needs capping or banding.

In [ ]:
entire = df_eda[df_eda["is_entire"]]

entire.groupby(["city", "guests"]).size().unstack().reindex(CITY_ORDER)

In [ ]:
entire.groupby(["city", "guests"])["price_num"].median().unstack().reindex(CITY_ORDER)

In [ ]:
pd.DataFrame({
    "all_listings": spearman_by_city(df_eda, "accommodates"),
    "entire_homes": spearman_by_city(entire, "accommodates"),
})

In [ ]:
guests_premium = to_pct(entire.groupby(["guests", "city"])["rel_log_entire"].median().unstack())[CITY_ORDER]

ax = guests_premium.plot(marker="o", figsize=(8, 4))
ax.axhline(0, color="grey", linewidth=0.8)
ax.set(title="Entire homes: median price vs the city's entire-home median, by capacity",
       xlabel="Guests (10 = 10 or more)", ylabel="% vs city median")
ax.legend(title="City")
plt.tight_layout()
plt.savefig(FIG_DIR / "b3_price_by_capacity.png", dpi=150)
plt.show()

### Findings

- `accommodates` is complete and ranges from 1 to 16 (median 3, p99 11) among active listings (Track A, A2). Capacity is capped at 10 here (`guests`) only to avoid medians based on a handful of very large listings.

> **Pending execution:** record the Spearman correlations (all listings vs entire homes) and describe the chart: the % premium per extra guest, the capacity at which the curve flattens, and whether the shape is similar across cities. A higher correlation for "all listings" than for "entire homes" means part of the capacity effect is a room-type effect.

### Conclusion

- `accommodates` is a core feature. If the curve flattens above a capacity level, capping it at that level on Day 2 reduces the influence of rare large listings.
- Because the price-capacity curve differs by city, a tree-based model (which learns interactions such as city x capacity automatically) is a natural candidate.

## B4. Bedrooms, beds and bathrooms

**Research question:** Are bedrooms, beds and bathrooms related to price beyond what guest capacity already captures?

**Purpose:** Size features are strongly correlated with each other. If they add nothing beyond `accommodates`, they add missing-value handling and noise without improving the model. `bathrooms_text` is parsed here into a count and a shared flag for exploration (Track A, A4.1); the final parsing is a Day 2 step.

In [ ]:
bath_text = df_eda["bathrooms_text"].str.lower()

df_eda["bath_count"] = bath_text.str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
is_half_bath = bath_text.str.contains("half-bath", na=False) & df_eda["bath_count"].isna()
df_eda.loc[is_half_bath, "bath_count"] = 0.5
df_eda["bath_shared"] = bath_text.str.contains("shared", na=False)

df_eda[["bath_count", "bath_shared"]].describe(include="all")

In [ ]:
SIZE_FEATURES = ["accommodates", "bedrooms", "beds", "bath_count"]
entire = df_eda[df_eda["is_entire"]]

size_correlation = entire[SIZE_FEATURES + ["rel_log_entire"]].corr(method="spearman")
size_correlation

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(size_correlation, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_title("Entire homes: Spearman correlation of size features and relative price")
plt.tight_layout()
plt.savefig(FIG_DIR / "b4_size_feature_correlation.png", dpi=150)
plt.show()

In [ ]:
pd.DataFrame({feature: spearman_by_city(entire, feature) for feature in SIZE_FEATURES})

In [ ]:
pd.DataFrame({
    feature: spearman_by_city(entire, feature, target="rel_log_size")
    for feature in ["bedrooms", "beds", "bath_count"]
})

In [ ]:
private_rooms = df_eda[df_eda["room_type"] == "Private room"]

median_premium_pct(private_rooms, "bath_shared", value="rel_log")

### Findings

- Track C showed that `bathrooms` is 6.6% missing among active listings while `bathrooms_text` is only 0.1% missing, so the parsed `bath_count` is the more complete bathroom feature.

> **Pending execution:** record:
1. the size correlation matrix - correlations above ~0.8 between two features mean they carry nearly the same information;
2. the correlation of `bedrooms`, `beds` and `bath_count` with `rel_log_size` - this measures what each feature adds **at the same guest capacity**. Values near 0 mean "no added information"; clearly positive values mean the feature should be kept;
3. the shared-bath difference for private rooms (median relative price with vs without a shared bath).

### Conclusion

- Decision rule for Day 2: keep a size feature only if it is related to price **at the same capacity** in most cities (point 2 above); otherwise it adds missing-value handling without information.
- Use the parsed `bath_count` and `bath_shared` instead of the numeric `bathrooms` column.

## B5. Property type

**Research question:** Do property types differ in price at the same size, and can the 100 property types be grouped without losing information?

**Purpose:** `property_type` has 100 categories, many with very few listings (Track A, A3). One-hot encoding them creates sparse, unstable columns. If a handful of groups captures almost the same variance, grouping is the better Day 2 choice.

In [ ]:
entire = df_eda[df_eda["is_entire"]]

entire["property_type"].value_counts().head(12).rename("entire_home_listings").to_frame()

In [ ]:
property_premium = entire.groupby("property_type").agg(
    listings=("rel_log_size", "size"),
    median_rel_log_size=("rel_log_size", "median"),
)
property_premium["premium_%"] = to_pct(property_premium["median_rel_log_size"])
property_premium = property_premium.query("listings >= 200").sort_values("premium_%")
property_premium

In [ ]:
ax = property_premium["premium_%"].plot(kind="barh", figsize=(8, 5), color="#4C72B0")
ax.axvline(0, color="grey", linewidth=0.8)
ax.set(title="Entire homes: price vs same-city, same-capacity median, by property type (>= 200 listings)",
       xlabel="% vs comparable listings", ylabel="")
plt.tight_layout()
plt.savefig(FIG_DIR / "b5_property_type_premium.png", dpi=150)
plt.show()

In [ ]:
PROPERTY_GROUP_KEYWORDS = [
    ("hotel / hostel / B&B", ["hotel", "hostel", "bed and breakfast", "guesthouse"]),
    ("serviced apartment", ["serviced apartment"]),
    ("house / villa", ["home", "house", "townhouse", "villa", "cottage", "bungalow"]),
    ("condo / loft", ["condo", "loft"]),
    ("rental unit", ["rental unit"]),
]


def group_property_type(property_type: str) -> str:
    text = str(property_type).lower()
    for group, keywords in PROPERTY_GROUP_KEYWORDS:
        if any(keyword in text for keyword in keywords):
            return group
    return "other"


df_eda["property_group"] = df_eda["property_type"].map(group_property_type)
pd.crosstab(df_eda["property_group"], df_eda["room_type"])

In [ ]:
pd.DataFrame({
    "property_type (100 values)": variance_explained(df_eda, "property_type"),
    "property_group (6 values)": variance_explained(df_eda, "property_group"),
    "room_type (4 values)": variance_explained(df_eda, "room_type"),
})

### Findings

- The most common active property type is "Entire rental unit" (106,351 listings, Track A); most other types are small.

> **Pending execution:** record:
1. the property types with the largest positive and negative premiums at the same city and capacity;
2. the variance-explained table. If `property_group` explains almost as much as `property_type` (and clearly more than `room_type` alone), the grouping keeps the information with far fewer categories. Check the crosstab for group/room-type combinations that look wrong before adopting the mapping.

### Conclusion

- **Proposed (pending the result above):** replace `property_type` with `property_group` on Day 2. The keyword mapping is a first draft and should be reviewed by the team.

## B6. Location

**Research question:** How much of the price variation within a city is explained by location - by neighbourhood and by distance from the city centre?

**Purpose:** Location is usually the strongest price driver after size and room type. Comparing a neighbourhood encoding with a simple distance-to-centre measure shows which location features Day 2 should build. The centre points are conventional city-centre coordinates (an assumption, not a data-driven choice). Coordinates are shifted by up to ~150 m by Inside Airbnb for privacy, which does not matter at this scale.

In [ ]:
CITY_CENTRE = {
    "london": (51.5074, -0.1278),
    "paris": (48.8566, 2.3522),
    "rome": (41.9028, 12.4964),
    "madrid": (40.4168, -3.7038),
    "barcelona": (41.3870, 2.1701),
}


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


centre_lat = df_eda["city"].map({city: point[0] for city, point in CITY_CENTRE.items()})
centre_lon = df_eda["city"].map({city: point[1] for city, point in CITY_CENTRE.items()})
df_eda["distance_km"] = haversine_km(df_eda["latitude"], df_eda["longitude"], centre_lat, centre_lon)

df_eda.groupby("city")["distance_km"].describe(percentiles=[0.5, 0.9]).reindex(CITY_ORDER)

In [ ]:
entire = df_eda[df_eda["is_entire"]]

pd.DataFrame({
    "all_listings": spearman_by_city(df_eda, "distance_km"),
    "entire_homes_same_size": spearman_by_city(entire, "distance_km", target="rel_log_size"),
})

In [ ]:
DISTANCE_BINS = [0, 1, 2, 4, 8, np.inf]
DISTANCE_LABELS = ["0-1 km", "1-2 km", "2-4 km", "4-8 km", "8+ km"]

df_eda["distance_band"] = pd.cut(df_eda["distance_km"], DISTANCE_BINS, labels=DISTANCE_LABELS, include_lowest=True)
entire = df_eda[df_eda["is_entire"]]

distance_premium = median_premium_pct(entire, "distance_band", value="rel_log_size")
distance_premium

In [ ]:
ax = distance_premium.T.plot(marker="o", figsize=(8, 4))
ax.axhline(0, color="grey", linewidth=0.8)
ax.set(title="Entire homes: price vs same-city, same-capacity median, by distance from centre",
       xlabel="Distance from city centre", ylabel="% vs comparable listings")
ax.legend(title="City")
plt.tight_layout()
plt.savefig(FIG_DIR / "b6_price_by_distance.png", dpi=150)
plt.show()

In [ ]:
neighbourhood_summary = pd.DataFrame({
    "neighbourhoods": df_eda.groupby("city")["neighbourhood_cleansed"].nunique(),
    "median_listings_per_neighbourhood": df_eda.groupby(["city", "neighbourhood_cleansed"]).size().groupby(level="city").median(),
    "eta2_neighbourhood": variance_explained(df_eda, "neighbourhood_cleansed"),
    "eta2_distance_band": variance_explained(df_eda, "distance_band"),
}).reindex(CITY_ORDER)
neighbourhood_summary

### Findings

- There are 265 official neighbourhoods across the five cities, with very different granularity per city (Track A, A3).

> **Pending execution:** record:
1. the Spearman correlation of distance with price (expected negative: further from the centre, cheaper) and whether it holds at the same size;
2. the distance-band premiums - where the price drop is steepest and whether any city deviates (for example, a second premium area away from the centre);
3. the neighbourhood table: number of neighbourhoods, listings per neighbourhood and the share of variance explained by neighbourhood vs distance band.

### Conclusion

- Day 2 location features (proposed): `distance_km` to the city centre plus a smoothed neighbourhood encoding fitted on the training folds only (to avoid target leakage). If neighbourhood explains much more variance than distance, the neighbourhood encoding is the priority.

## B7. Host characteristics

**Research question:** Are host characteristics - Superhost status, portfolio size and tenure - related to price at the same city, room type and capacity?

**Purpose:** A new host knows their own profile, so these are legitimate features. Professional multi-listing operators may price differently from individual hosts; if so, the model needs a host-type feature.

In [ ]:
df_eda["is_superhost"] = df_eda["host_is_superhost"] == "t"
df_eda["host_band"] = pd.cut(
    df_eda["calculated_host_listings_count"], [0, 1, 5, 50, np.inf], labels=["1", "2-5", "6-50", "50+"]
)
entire = df_eda[df_eda["is_entire"]]

median_premium_pct(entire, "is_superhost", value="rel_log_size")

In [ ]:
host_band_premium = median_premium_pct(entire, "host_band", value="rel_log_size")
host_band_premium

In [ ]:
ax = host_band_premium.plot(kind="bar", figsize=(9, 4), rot=0)
ax.axhline(0, color="grey", linewidth=0.8)
ax.set(title="Entire homes: price vs same-city, same-capacity median, by host portfolio size",
       xlabel="", ylabel="% vs comparable listings")
ax.legend(title="Host's listings in city")
plt.tight_layout()
plt.savefig(FIG_DIR / "b7_price_by_host_size.png", dpi=150)
plt.show()

In [ ]:
spearman_by_city(entire, "hosts_time_as_host_years", target="rel_log_size").to_frame("tenure_vs_relative_price")

### Findings

- 29.5% of active listings belong to Superhosts (Track A). Hosts with 6+ listings supply 27-64% of active listings depending on the city (Track C, C5).

> **Pending execution:** record the Superhost premium per city, the host-band premiums and the tenure correlation. A consistent premium or discount for 50+ listing operators across cities supports a host-type feature; premiums below ~2-3% are too small to matter.

### Conclusion

- Proposed Day 2 host features: `host_is_superhost`, `calculated_host_listings_count` (log-transformed or banded) and `hosts_time_as_host_years`, kept only if the results above show consistent associations.
- Association, not causation: Superhost status may reflect listing quality rather than cause a higher price.

## B8. Amenities

**Research question:** Which amenities are associated with higher prices at the same city, room type and capacity, and does the number of amenities matter?

**Purpose:** Amenities are under the host's control and are known before pricing. Testing them against comparable listings (`rel_log_size`) avoids the trap of "large homes have more amenities and are more expensive". The result selects a small set of binary amenity flags for Day 2.

In [ ]:
def parse_amenities(text):
    try:
        return json.loads(text)
    except (TypeError, ValueError):
        return []


df_eda["amenity_list"] = df_eda["amenities"].map(parse_amenities)
df_eda["amenity_count"] = df_eda["amenity_list"].map(len)
df_eda.groupby("city")["amenity_count"].describe(percentiles=[0.5]).reindex(CITY_ORDER)

In [ ]:
entire = df_eda[df_eda["is_entire"]]

spearman_by_city(entire, "amenity_count", target="rel_log_size").to_frame("amenity_count_vs_relative_price")

In [ ]:
amenity_sets = entire["amenity_list"].map(set)
amenity_prevalence = entire["amenity_list"].explode().value_counts() / len(entire)

candidate_amenities = amenity_prevalence[amenity_prevalence.between(0.10, 0.90)].index
print(f"{len(candidate_amenities)} amenities are present in 10-90% of entire homes")

In [ ]:
def amenity_premium(amenity: str) -> pd.Series:
    has_amenity = amenity_sets.map(lambda items: amenity in items)
    median_with = entire.loc[has_amenity, "rel_log_size"].median()
    median_without = entire.loc[~has_amenity, "rel_log_size"].median()
    city_gaps = (
        entire.groupby([entire["city"], has_amenity.rename("has")])["rel_log_size"]
        .median().unstack()
    )
    city_gaps = city_gaps.reindex(columns=[False, True])
    return pd.Series({
        "prevalence_%": amenity_prevalence[amenity] * 100,
        "premium_%": to_pct(median_with - median_without),
        "cities_with_positive_gap": int((city_gaps[True] > city_gaps[False]).sum()),
    })


amenity_premiums = pd.DataFrame({amenity: amenity_premium(amenity) for amenity in candidate_amenities}).T
amenity_premiums = amenity_premiums.sort_values("premium_%", ascending=False)
amenity_premiums.head(15)

In [ ]:
amenity_premiums.tail(10)

In [ ]:
strongest = amenity_premiums.reindex(amenity_premiums["premium_%"].abs().nlargest(20).index).sort_values("premium_%")

ax = strongest["premium_%"].plot(kind="barh", figsize=(8, 7), color="#4C72B0")
ax.axvline(0, color="grey", linewidth=0.8)
ax.set(title="Entire homes: price gap with vs without each amenity (20 largest gaps)",
       xlabel="% difference vs comparable listings", ylabel="")
plt.tight_layout()
plt.savefig(FIG_DIR / "b8_amenity_premiums.png", dpi=150)
plt.show()

### Findings

- Amenities are stored as a JSON list inside a string and parse with `json.loads` (Track A, A4.2).

> **Pending execution:** record:
1. the correlation of `amenity_count` with relative price per city;
2. the top and bottom amenities, keeping only those with a consistent direction in at least 4 of 5 cities (`cities_with_positive_gap` >= 4 for a premium, <= 1 for a discount);
3. the number of candidate amenities (present in 10-90% of entire homes).

### Conclusion

- Proposed Day 2 features: `amenity_count` plus 10-15 binary flags for the amenities with the largest **and** most consistent gaps.
- Association, not causation: an amenity such as air conditioning may mark a renovated or premium listing rather than raise the price by itself. HostWise should describe these as "listings with X tend to be priced higher", not as advice that adding X raises the market price.

## B9. Reviews

**Research question:** Are review scores and review volume related to price at the same city, room type and capacity?

**Purpose:** Track A showed a strong ceiling effect (median rating 4.82). If ratings barely separate listings, they add little; if listings without reviews are priced differently, a `has_no_reviews` flag is useful. A new host has no reviews, so this also shows how the model will treat new listings.

In [ ]:
REVIEW_BAND_ORDER = ["no reviews", "<=4.50", "4.51-4.70", "4.71-4.85", "4.86-5.00"]

score_band = pd.cut(
    df_eda["review_scores_rating"], bins=[0, 4.5, 4.7, 4.85, 5.0],
    labels=REVIEW_BAND_ORDER[1:], include_lowest=True,
)
df_eda["review_band"] = pd.Categorical(
    score_band.astype(object).where(df_eda["number_of_reviews"] > 0, "no reviews"),
    categories=REVIEW_BAND_ORDER,
)
pd.crosstab(df_eda["city"], df_eda["review_band"], normalize="index").mul(100).reindex(CITY_ORDER)

In [ ]:
entire = df_eda[df_eda["is_entire"]]

review_premium = median_premium_pct(entire, "review_band", value="rel_log_size")
review_premium

In [ ]:
ax = review_premium.plot(kind="bar", figsize=(9, 4), rot=0)
ax.axhline(0, color="grey", linewidth=0.8)
ax.set(title="Entire homes: price vs same-city, same-capacity median, by review band",
       xlabel="", ylabel="% vs comparable listings")
ax.legend(title="Rating band")
plt.tight_layout()
plt.savefig(FIG_DIR / "b9_price_by_review_band.png", dpi=150)
plt.show()

In [ ]:
pd.DataFrame({
    feature: spearman_by_city(entire, feature, target="rel_log_size")
    for feature in ["review_scores_rating", "review_scores_location", "number_of_reviews", "reviews_per_month"]
})

### Findings

- Missing review scores occur exactly for listings with no reviews (Track C, C8), so "no reviews" is a separate band here, not a missing value. A score of 0 is placed in the lowest band (`include_lowest=True`); the earlier version labelled such scores as "no reviews" by mistake.
- `review_scores_rating` has median 4.82 and p1 3.00 among active listings (Track A): most listings sit in a narrow band near 5.

> **Pending execution:** record the share of listings per review band, the price premium of each band and the correlations. Check in particular: (1) whether "no reviews" listings are priced above or below reviewed listings; (2) whether `review_scores_location` is more related to price than the overall rating (it partly measures location, which B6 covers); (3) the sign of `number_of_reviews` - a negative correlation would suggest cheaper listings get booked (and reviewed) more, i.e. reverse causality.

### Conclusion

- Proposed Day 2 review features: `has_no_reviews` flag, `review_scores_rating` and `review_scores_location` with a neutral fill for listings without reviews, and `log1p(number_of_reviews)`.
- Review volume reflects past bookings at past prices; it should be interpreted as a popularity signal, not as a cause of price.

## B10. Which features explain the most price variance?

**Research question:** Ranked side by side, which features explain the largest share of the log-price variance within each city?

**Purpose:** This summarises B2-B9 in one comparable number per feature and city (eta squared, `variance_explained`). It sets the Day 2 feature priorities. Limitations: each feature is measured on its own, so overlapping features (for example, capacity and bedrooms) cannot be added up, and features with many categories (neighbourhood) are slightly favoured.

In [ ]:
df_eda["bedrooms_capped"] = df_eda["bedrooms"].clip(upper=5)
df_eda["bath_count_capped"] = df_eda["bath_count"].clip(upper=4)
df_eda["amenity_count_band"] = pd.qcut(df_eda["amenity_count"], 5, duplicates="drop")

ETA_FEATURES = [
    "room_type", "guests", "bedrooms_capped", "bath_count_capped", "property_group",
    "neighbourhood_cleansed", "distance_band", "is_superhost", "host_band",
    "review_band", "amenity_count_band",
]

eta_squared = pd.DataFrame({feature: variance_explained(df_eda, feature) for feature in ETA_FEATURES}).T
eta_squared["macro_average"] = eta_squared.mean(axis=1)
eta_squared = eta_squared.sort_values("macro_average", ascending=False)
eta_squared

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(eta_squared, annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set(title="Share of within-city log-price variance explained by each feature (eta squared)",
       xlabel="", ylabel="")
plt.tight_layout()
plt.savefig(FIG_DIR / "b10_variance_explained.png", dpi=150)
plt.show()

### Findings

> **Pending execution:** record the ranking (macro-average column) and any city where the order differs strongly. Expected from the earlier sections: room type, capacity and location at the top; host and review features at the bottom.

### Conclusion

- Features with a high eta squared in all cities are the core of the v1 model. Features with a very low value in every city (for example below 0.01) are candidates to drop, unless B2-B9 showed a clear effect at the same size.
- Because the ranking differs by city, per-city error reporting (and possibly city-specific models) should be compared on Day 2.

## Track B summary

**Confirmed**
- The EDA now uses the short-term modelling population with only extreme prices removed, and all price comparisons are made within cities or with currency-free relative measures.
- The raw price is strongly right-skewed in every city -> model `log_price`.
- The short-term scope changes the picture for Barcelona: its median price is 255 EUR in scope vs 181 EUR across all active priced listings.
- Missing review scores mean "no reviews"; private rooms are cheaper and more often lack `bedrooms`.

**To complete after execution** (each section's Findings lists the exact outputs): room-type gaps (B2), capacity curve (B3), added value of size features (B4), property-type grouping (B5), location (B6), host (B7), amenities (B8), reviews (B9) and the variance ranking (B10).

**Proposed Day 2 feature set (pending the results above and team approval)**

| Group | Features |
|---|---|
| Market | `city` |
| Listing type | `room_type`, `property_group` |
| Size | `accommodates`, plus `bedrooms`, `bath_count`, `bath_shared` if B4 shows added value |
| Location | `distance_km`, smoothed neighbourhood encoding (fitted on training folds) |
| Host | `host_is_superhost`, `calculated_host_listings_count`, `hosts_time_as_host_years` |
| Amenities | `amenity_count` + 10-15 selected flags |
| Reviews | `has_no_reviews`, `review_scores_rating`, `review_scores_location`, `log1p(number_of_reviews)` |

Excluded: target-leakage columns, borderline demand columns (pending), identifiers, empty columns and free text (Track A, A5).